Imports


In [55]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report,confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier                                                  
from lightgbm import LGBMClassifier                                                

In [56]:
df=pd.read_csv('../Data/polish_5year.csv')

In [57]:
df = df.drop(
    columns=["current_assets_minus_inventories_long_term_liabilities"]
)

Training


In [58]:
X = df.drop(columns=["class"])
y = df["class"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline([
    
    # Missing values
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    # Scaling
    (
        "scaler",
        StandardScaler()
    ),

    # Logistic Regression
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])



model.fit(
    X_train,
    y_train
)



y_pred = model.predict(X_test)

In [59]:
# ============================================================
# 8. F1 SCORE
# ============================================================

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1
)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

print("\nF1 Score:", f1)


# ============================================================
# 9. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)



MODEL PERFORMANCE

F1 Score: 0.33432835820895523

Classification Report:
              precision    recall  f1-score   support

           0     0.9720    0.8209    0.8901      1100
           1     0.2213    0.6829    0.3343        82

    accuracy                         0.8113      1182
   macro avg     0.5967    0.7519    0.6122      1182
weighted avg     0.9199    0.8113    0.8515      1182



### Models Evaluated

- HistGradientBoosting
- Random Forest
- XGBoost
- LightGBM

In [61]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# =========================================================
# 1. Load data
# =========================================================

df = pd.read_csv("../Data/polish_5year.csv")

# Remove X37
removed_feature = "current_assets_minus_inventories_long_term_liabilities"

df = df.drop(columns=[removed_feature])

X = df.drop(columns=["class"])
y = df["class"]


# =========================================================
# 2. Train / Test Split
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# =========================================================
# 3. Imputation
# =========================================================

imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)


# =========================================================
# 4. Models
# =========================================================

models = {

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42,
        verbosity=-1
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}


# =========================================================
# 5. Train + Evaluate
# =========================================================

results = []

for name, model in models.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Probabilities
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)

    print(f"F1 Score:        {f1:.4f}")
    print(f"Precision:       {precision:.4f}")
    print(f"Recall:          {recall:.4f}")
    print(f"ROC-AUC:         {roc_auc:.4f}")
    print(f"PR-AUC:          {pr_auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    results.append({
        "Model": name,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    })


# =========================================================
# 6. Compare Models
# =========================================================

results_df = pd.DataFrame(results)

print("\n\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    results_df
    .sort_values("F1", ascending=False)
    .to_string(index=False)
)


XGBoost
F1 Score:        0.6515
Precision:       0.8600
Recall:          0.5244
ROC-AUC:         0.9490
PR-AUC:          0.7529

Classification Report:
              precision    recall  f1-score   support

           0     0.9655    0.9936    0.9794      1100
           1     0.8600    0.5244    0.6515        82

    accuracy                         0.9611      1182
   macro avg     0.9128    0.7590    0.8155      1182
weighted avg     0.9582    0.9611    0.9566      1182

Confusion Matrix:
[[1093    7]
 [  39   43]]

HistGradientBoosting
F1 Score:        0.6917
Precision:       0.9020
Recall:          0.5610
ROC-AUC:         0.9649
PR-AUC:          0.7989

Classification Report:
              precision    recall  f1-score   support

           0     0.9682    0.9955    0.9816      1100
           1     0.9020    0.5610    0.6917        82

    accuracy                         0.9653      1182
   macro avg     0.9351    0.7782    0.8367      1182
weighted avg     0.9636    0.9653    

After add Cross Validation

In [62]:
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# =========================================================
# 1. Load data
# =========================================================

df = pd.read_csv("../Data/polish_5year.csv")

# Remove X37
removed_feature = "current_assets_minus_inventories_long_term_liabilities"

df = df.drop(columns=[removed_feature])

X = df.drop(columns=["class"])
y = df["class"]


# =========================================================
# 2. Train / Test Split
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# =========================================================
# 3. Models
# =========================================================

models = {

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42,
        verbosity=-1
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}


# =========================================================
# 4. 5-Fold Stratified Cross Validation
# =========================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


scoring = {
    "F1": "f1",
    "Precision": "precision",
    "Recall": "recall",
    "ROC-AUC": "roc_auc",
    "PR-AUC": "average_precision"
}


cv_results = []


for name, model in models.items():

    print("\n" + "=" * 70)
    print(f"{name} - 5-FOLD CROSS VALIDATION")
    print("=" * 70)

    # Imputer MUST be inside the pipeline
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    result = {
        "Model": name,

        "F1 Mean": scores["test_F1"].mean(),
        "F1 Std": scores["test_F1"].std(),

        "Precision Mean": scores["test_Precision"].mean(),
        "Recall Mean": scores["test_Recall"].mean(),

        "ROC-AUC Mean": scores["test_ROC-AUC"].mean(),
        "PR-AUC Mean": scores["test_PR-AUC"].mean()
    }

    cv_results.append(result)

    print(f"F1:        {result['F1 Mean']:.4f} ± {result['F1 Std']:.4f}")
    print(f"Precision: {result['Precision Mean']:.4f}")
    print(f"Recall:    {result['Recall Mean']:.4f}")
    print(f"ROC-AUC:   {result['ROC-AUC Mean']:.4f}")
    print(f"PR-AUC:    {result['PR-AUC Mean']:.4f}")


# =========================================================
# 5. CV Comparison
# =========================================================

cv_results_df = pd.DataFrame(cv_results)

print("\n\n" + "=" * 70)
print("5-FOLD CROSS VALIDATION COMPARISON")
print("=" * 70)

print(
    cv_results_df
    .sort_values("F1 Mean", ascending=False)
    .to_string(index=False)
)


# =========================================================
# 6. Final Test Evaluation
# =========================================================

print("\n\n" + "=" * 70)
print("FINAL TEST SET EVALUATION")
print("=" * 70)


test_results = []


for name, model in models.items():

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", model)
    ])

    # Train on the complete training set
    pipeline.fit(X_train, y_train)

    # Predict test set
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Metrics
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)

    print("\n" + "-" * 70)
    print(name)
    print("-" * 70)

    print(f"F1 Score:        {f1:.4f}")
    print(f"Precision:       {precision:.4f}")
    print(f"Recall:          {recall:.4f}")
    print(f"ROC-AUC:         {roc_auc:.4f}")
    print(f"PR-AUC:          {pr_auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    test_results.append({
        "Model": name,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    })


# =========================================================
# 7. Final Test Comparison
# =========================================================

test_results_df = pd.DataFrame(test_results)

print("\n\n" + "=" * 70)
print("FINAL TEST COMPARISON")
print("=" * 70)

print(
    test_results_df
    .sort_values("F1", ascending=False)
    .to_string(index=False)
)


XGBoost - 5-FOLD CROSS VALIDATION
F1:        0.6589 ± 0.0637
Precision: 0.9142
Recall:    0.5216
ROC-AUC:   0.9468
PR-AUC:    0.7791

HistGradientBoosting - 5-FOLD CROSS VALIDATION
F1:        0.6931 ± 0.0458
Precision: 0.9215
Recall:    0.5611
ROC-AUC:   0.9421
PR-AUC:    0.7741

LightGBM - 5-FOLD CROSS VALIDATION
F1:        0.7063 ± 0.0408
Precision: 0.8149
Recall:    0.6250
ROC-AUC:   0.9382
PR-AUC:    0.7585

Random Forest - 5-FOLD CROSS VALIDATION
F1:        0.3473 ± 0.0509
Precision: 0.6507
Recall:    0.2376
ROC-AUC:   0.8952
PR-AUC:    0.4759


5-FOLD CROSS VALIDATION COMPARISON
               Model  F1 Mean   F1 Std  Precision Mean  Recall Mean  ROC-AUC Mean  PR-AUC Mean
            LightGBM 0.706311 0.040820        0.814942     0.625035      0.938247     0.758460
HistGradientBoosting 0.693075 0.045761        0.921455     0.561072      0.942055     0.774091
             XGBoost 0.658911 0.063695        0.914168     0.521585      0.946771     0.779142
       Random Forest 0.3472

## 🥇 Best Baseline Model — LightGBM


### Hyperparameter Tuning

Testing multiple hyperparameter configurations for LightGBM to identify the best-performing setup.

In [63]:
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from lightgbm import LGBMClassifier


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("../Data/polish_5year.csv")

print("Original shape:", df.shape)


# ============================================================
# 2. REMOVE X37
# ============================================================

removed_feature = "current_assets_minus_inventories_long_term_liabilities"

df = df.drop(columns=[removed_feature])

print("Shape after removing X37:", df.shape)


# ============================================================
# 3. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop(columns=["class"])
y = df["class"]

print("\nClass distribution:")
print(y.value_counts())


# ============================================================
# 4. TRAIN / TEST SPLIT
# ============================================================

# 80% Training
# 20% Untouched Test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Test samples:", len(X_test))


# ============================================================
# 5. PIPELINE
# ============================================================

# IMPORTANT:
# Imputation happens inside the CV pipeline
# so there is no data leakage between folds.

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),

    ("classifier", LGBMClassifier(
        objective="binary",
        class_weight="balanced",
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    ))
])


# ============================================================
# 6. HYPERPARAMETER SEARCH SPACE
# ============================================================

param_grid = {

    "classifier__n_estimators": [
        100,
        200,
        300,
        500,
        700
    ],

    "classifier__learning_rate": [
        0.01,
        0.03,
        0.05,
        0.08,
        0.1
    ],

    "classifier__num_leaves": [
        15,
        31,
        50,
        70,
        100
    ],

    "classifier__max_depth": [
        -1,
        5,
        7,
        10,
        15
    ],

    "classifier__min_child_samples": [
        10,
        20,
        30,
        50,
        100
    ],

    "classifier__subsample": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "classifier__colsample_bytree": [
        0.7,
        0.8,
        0.9,
        1.0
    ]
}


# ============================================================
# 7. 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# 8. RANDOMIZED SEARCH
# ============================================================

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,

    n_iter=50,

    scoring="f1",

    cv=cv,

    random_state=42,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)


# ============================================================
# 9. RUN TUNING
# ============================================================

print("\n" + "=" * 70)
print("STARTING LIGHTGBM HYPERPARAMETER TUNING")
print("=" * 70)

search.fit(X_train, y_train)


# ============================================================
# 10. BEST RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BEST LIGHTGBM PARAMETERS")
print("=" * 70)

print("\nBest CV F1:")
print(search.best_score_)

print("\nBest Parameters:")

for parameter, value in search.best_params_.items():
    print(f"{parameter}: {value}")


# ============================================================
# 11. BEST MODEL
# ============================================================

best_model = search.best_estimator_


# ============================================================
# 12. FINAL TEST SET EVALUATION
# ============================================================

y_pred = best_model.predict(X_test)

y_prob = best_model.predict_proba(X_test)[:, 1]


f1 = f1_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

pr_auc = average_precision_score(y_test, y_prob)


# ============================================================
# 13. PRINT TEST RESULTS
# ============================================================

print("\n" + "=" * 70)
print("TUNED LIGHTGBM - FINAL TEST SET")
print("=" * 70)

print(f"\nF1 Score:        {f1:.4f}")
print(f"Precision:       {precision:.4f}")
print(f"Recall:          {recall:.4f}")
print(f"ROC-AUC:         {roc_auc:.4f}")
print(f"PR-AUC:          {pr_auc:.4f}")


# ============================================================
# 14. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)


# ============================================================
# 15. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")

print(cm)

Original shape: (5910, 65)
Shape after removing X37: (5910, 64)

Class distribution:
class
0    5500
1     410
Name: count, dtype: int64

Training samples: 4728
Test samples: 1182

STARTING LIGHTGBM HYPERPARAMETER TUNING
Fitting 5 folds for each of 50 candidates, totalling 250 fits

BEST LIGHTGBM PARAMETERS

Best CV F1:
0.7129710010248915

Best Parameters:
classifier__subsample: 0.9
classifier__num_leaves: 70
classifier__n_estimators: 700
classifier__min_child_samples: 20
classifier__max_depth: 15
classifier__learning_rate: 0.1
classifier__colsample_bytree: 0.9

TUNED LIGHTGBM - FINAL TEST SET

F1 Score:        0.6861
Precision:       0.8545
Recall:          0.5732
ROC-AUC:         0.9615
PR-AUC:          0.8073

Classification Report:
              precision    recall  f1-score   support

           0     0.9689    0.9927    0.9807      1100
           1     0.8545    0.5732    0.6861        82

    accuracy                         0.9636      1182
   macro avg     0.9117    0.7829   

In [64]:
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from lightgbm import LGBMClassifier


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("../Data/polish_5year.csv")

print("Original shape:", df.shape)


# ============================================================
# 2. REMOVE X37
# ============================================================

removed_feature = "current_assets_minus_inventories_long_term_liabilities"

df = df.drop(columns=[removed_feature])

print("Shape after removing X37:", df.shape)


# ============================================================
# 3. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop(columns=["class"])
y = df["class"]

print("\nClass distribution:")
print(y.value_counts())


# ============================================================
# 4. TRAIN / TEST SPLIT
# ============================================================

# 80% Training
# 20% Untouched Test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Test samples:", len(X_test))


# ============================================================
# 5. PIPELINE
# ============================================================

# IMPORTANT:
# Imputation happens inside the CV pipeline
# so there is no data leakage between folds.

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),

    ("classifier", LGBMClassifier(
        objective="binary",
        class_weight="balanced",
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    ))
])


# ============================================================
# 6. GRID SEARCH PARAMETER SPACE
# ============================================================

# Focused grid around reasonable LightGBM values.
#
# Total combinations:
#
# 3 n_estimators
# × 3 learning_rate
# × 3 num_leaves
# × 3 max_depth
#
# = 81 combinations
#
# With 5-fold CV:
#
# 81 × 5 = 405 fits

param_grid = {

    "classifier__n_estimators": [
        300,
        500,
        700
    ],

    "classifier__learning_rate": [
        0.03,
        0.05,
        0.1
    ],

    "classifier__num_leaves": [
        31,
        50,
        70
    ],

    "classifier__max_depth": [
        -1,
        7,
        10
    ]
}


# ============================================================
# 7. 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# 8. GRID SEARCH
# ============================================================

search = GridSearchCV(
    estimator=pipeline,

    param_grid=param_grid,

    scoring="f1",

    cv=cv,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)


# ============================================================
# 9. RUN GRID SEARCH
# ============================================================

print("\n" + "=" * 70)
print("STARTING LIGHTGBM GRID SEARCH")
print("=" * 70)

print("\nTotal parameter combinations:", 81)
print("Total CV fits:", 81 * 5)

search.fit(X_train, y_train)


# ============================================================
# 10. BEST RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BEST LIGHTGBM PARAMETERS")
print("=" * 70)

print("\nBest CV F1:")
print(f"{search.best_score_:.4f}")

print("\nBest Parameters:")

for parameter, value in search.best_params_.items():
    print(f"{parameter}: {value}")


# ============================================================
# 11. BEST MODEL
# ============================================================

best_model = search.best_estimator_


# ============================================================
# 12. FINAL TEST SET EVALUATION
# ============================================================

# IMPORTANT:
# X_test was never used during Grid Search.

y_pred = best_model.predict(X_test)

y_prob = best_model.predict_proba(X_test)[:, 1]


# ============================================================
# 13. CALCULATE METRICS
# ============================================================

f1 = f1_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

pr_auc = average_precision_score(
    y_test,
    y_prob
)


# ============================================================
# 14. PRINT TEST RESULTS
# ============================================================

print("\n" + "=" * 70)
print("GRID-SEARCHED LIGHTGBM - FINAL TEST SET")
print("=" * 70)

print(f"\nF1 Score:        {f1:.4f}")
print(f"Precision:       {precision:.4f}")
print(f"Recall:          {recall:.4f}")
print(f"ROC-AUC:         {roc_auc:.4f}")
print(f"PR-AUC:          {pr_auc:.4f}")


# ============================================================
# 15. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)


# ============================================================
# 16. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")

print(cm)


# ============================================================
# 17. COMPARISON WITH BASELINE
# ============================================================

baseline_f1 = 0.7172

print("\n" + "=" * 70)
print("COMPARISON WITH BASELINE LIGHTGBM")
print("=" * 70)

print(f"\nBaseline Test F1:     {baseline_f1:.4f}")
print(f"Grid Search Test F1: {f1:.4f}")
print(f"Difference:           {f1 - baseline_f1:+.4f}")

if f1 > baseline_f1:
    print("\nGrid Search IMPROVED the baseline.")
elif f1 < baseline_f1:
    print("\nGrid Search did NOT improve the baseline.")
else:
    print("\nGrid Search produced the same F1 as the baseline.")

Original shape: (5910, 65)
Shape after removing X37: (5910, 64)

Class distribution:
class
0    5500
1     410
Name: count, dtype: int64

Training samples: 4728
Test samples: 1182

STARTING LIGHTGBM GRID SEARCH

Total parameter combinations: 81
Total CV fits: 405
Fitting 5 folds for each of 81 candidates, totalling 405 fits

BEST LIGHTGBM PARAMETERS

Best CV F1:
0.7179

Best Parameters:
classifier__learning_rate: 0.1
classifier__max_depth: -1
classifier__n_estimators: 300
classifier__num_leaves: 31

GRID-SEARCHED LIGHTGBM - FINAL TEST SET

F1 Score:        0.7050
Precision:       0.8596
Recall:          0.5976
ROC-AUC:         0.9563
PR-AUC:          0.8135

Classification Report:
              precision    recall  f1-score   support

           0     0.9707    0.9927    0.9816      1100
           1     0.8596    0.5976    0.7050        82

    accuracy                         0.9653      1182
   macro avg     0.9152    0.7951    0.8433      1182
weighted avg     0.9630    0.9653    0

In [53]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, precision_score, recall_score


# ============================================================
# THRESHOLD OPTIMIZATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Get out-of-fold probabilities from the tuned model
oof_prob = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]


# ============================================================
# TEST DIFFERENT THRESHOLDS
# ============================================================

thresholds = np.arange(0.10, 0.91, 0.01)

results = []

for threshold in thresholds:

    oof_pred = (oof_prob >= threshold).astype(int)

    f1 = f1_score(y_train, oof_pred)
    precision = precision_score(y_train, oof_pred, zero_division=0)
    recall = recall_score(y_train, oof_pred, zero_division=0)

    results.append({
        "Threshold": threshold,
        "F1": f1,
        "Precision": precision,
        "Recall": recall
    })


results_df = pd.DataFrame(results)


# ============================================================
# BEST THRESHOLD FOR F1
# ============================================================

best_row = results_df.loc[
    results_df["F1"].idxmax()
]

print("\n" + "=" * 70)
print("BEST THRESHOLD FOR F1")
print("=" * 70)

print(f"\nThreshold: {best_row['Threshold']:.2f}")
print(f"F1:        {best_row['F1']:.4f}")
print(f"Precision: {best_row['Precision']:.4f}")
print(f"Recall:    {best_row['Recall']:.4f}")


# ============================================================
# SHOW TOP 10 THRESHOLDS
# ============================================================

print("\n" + "=" * 70)
print("TOP 10 THRESHOLDS")
print("=" * 70)

print(
    results_df
    .sort_values("F1", ascending=False)
    .head(10)
    .to_string(index=False)
)


# ============================================================
# FINAL TEST EVALUATION
# ============================================================

best_threshold = best_row["Threshold"]

test_prob = best_model.predict_proba(X_test)[:, 1]

test_pred = (
    test_prob >= best_threshold
).astype(int)


test_f1 = f1_score(y_test, test_pred)
test_precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)
test_recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)


print("\n" + "=" * 70)
print("TUNED LIGHTGBM + OPTIMIZED THRESHOLD")
print("=" * 70)

print(f"\nThreshold: {best_threshold:.2f}")
print(f"F1:        {test_f1:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")


BEST THRESHOLD FOR F1

Threshold: 0.54
F1:        0.7174
Precision: 0.8839
Recall:    0.6037

TOP 10 THRESHOLDS
 Threshold       F1  Precision   Recall
      0.54 0.717391   0.883929 0.603659
      0.55 0.717391   0.883929 0.603659
      0.53 0.714801   0.876106 0.603659
      0.50 0.714542   0.868996 0.606707
      0.49 0.713262   0.865217 0.606707
      0.56 0.712727   0.882883 0.597561
      0.51 0.712230   0.868421 0.603659
      0.52 0.712230   0.868421 0.603659
      0.46 0.711986   0.861472 0.606707
      0.47 0.711986   0.861472 0.606707

TUNED LIGHTGBM + OPTIMIZED THRESHOLD

Threshold: 0.54
F1:        0.6861
Precision: 0.8545
Recall:    0.5732


In [54]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from lightgbm import LGBMClassifier


# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("../Data/polish_5year.csv")

removed_feature = "current_assets_minus_inventories_long_term_liabilities"

df = df.drop(columns=[removed_feature])

X = df.drop(columns=["class"])
y = df["class"]


# ============================================================
# 2. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 3. 5-FOLD STRATIFIED CV
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# 4. LIGHTGBM CONFIGURATIONS
# ============================================================

models = {

    "No Class Weight": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        class_weight=None,
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    ),

    "Balanced": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        class_weight="balanced",
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    ),

    "Custom 5:1": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        class_weight={0: 1, 1: 5},
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    ),

    "Custom 10:1": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        class_weight={0: 1, 1: 10},
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    ),

    "Custom 15:1": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        class_weight={0: 1, 1: 15},
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    )
}


# ============================================================
# 5. CROSS-VALIDATION
# ============================================================

scoring = {
    "F1": "f1",
    "Precision": "precision",
    "Recall": "recall",
    "ROC-AUC": "roc_auc",
    "PR-AUC": "average_precision"
}


cv_results = []


print("\n" + "=" * 70)
print("CLASS IMBALANCE EXPERIMENT")
print("=" * 70)


for name, classifier in models.items():

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", classifier)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    result = {
        "Model": name,
        "CV F1": scores["test_F1"].mean(),
        "F1 Std": scores["test_F1"].std(),
        "CV Precision": scores["test_Precision"].mean(),
        "CV Recall": scores["test_Recall"].mean(),
        "CV ROC-AUC": scores["test_ROC-AUC"].mean(),
        "CV PR-AUC": scores["test_PR-AUC"].mean()
    }

    cv_results.append(result)

    print("\n" + "-" * 70)
    print(name)
    print("-" * 70)

    print(f"F1:        {result['CV F1']:.4f} ± {result['F1 Std']:.4f}")
    print(f"Precision: {result['CV Precision']:.4f}")
    print(f"Recall:    {result['CV Recall']:.4f}")
    print(f"ROC-AUC:   {result['CV ROC-AUC']:.4f}")
    print(f"PR-AUC:    {result['CV PR-AUC']:.4f}")


# ============================================================
# 6. CV COMPARISON
# ============================================================

cv_df = pd.DataFrame(cv_results)

cv_df = cv_df.sort_values(
    "CV F1",
    ascending=False
)

print("\n" + "=" * 70)
print("CLASS IMBALANCE - CV COMPARISON")
print("=" * 70)

print(
    cv_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 7. SELECT BEST CONFIGURATION
# ============================================================

best_name = cv_df.iloc[0]["Model"]

best_classifier = models[best_name]

print("\n" + "=" * 70)
print("BEST CLASS-WEIGHT CONFIGURATION")
print("=" * 70)

print(f"\nBest configuration: {best_name}")
print(f"Best CV F1: {cv_df.iloc[0]['CV F1']:.4f}")


# ============================================================
# 8. FIT BEST CONFIGURATION ON FULL TRAINING SET
# ============================================================

best_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", best_classifier)
])

best_pipeline.fit(
    X_train,
    y_train
)


# ============================================================
# 9. FINAL TEST SET
# ============================================================

y_pred = best_pipeline.predict(X_test)

y_prob = best_pipeline.predict_proba(X_test)[:, 1]


test_f1 = f1_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_roc_auc = roc_auc_score(y_test, y_prob)
test_pr_auc = average_precision_score(y_test, y_prob)


# ============================================================
# 10. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BEST CLASS-WEIGHT CONFIGURATION - FINAL TEST")
print("=" * 70)

print(f"\nConfiguration: {best_name}")
print(f"F1:        {test_f1:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"ROC-AUC:   {test_roc_auc:.4f}")
print(f"PR-AUC:    {test_pr_auc:.4f}")


# ============================================================
# 11. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)


# ============================================================
# 12. CONFUSION MATRIX
# ============================================================

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


CLASS IMBALANCE EXPERIMENT

----------------------------------------------------------------------
No Class Weight
----------------------------------------------------------------------
F1:        0.6952 ± 0.0420
Precision: 0.9030
Recall:    0.5672
ROC-AUC:   0.9430
PR-AUC:    0.7811

----------------------------------------------------------------------
Balanced
----------------------------------------------------------------------
F1:        0.7017 ± 0.0353
Precision: 0.8095
Recall:    0.6220
ROC-AUC:   0.9369
PR-AUC:    0.7629

----------------------------------------------------------------------
Custom 5:1
----------------------------------------------------------------------
F1:        0.7060 ± 0.0415
Precision: 0.8479
Recall:    0.6069
ROC-AUC:   0.9390
PR-AUC:    0.7677

----------------------------------------------------------------------
Custom 10:1
----------------------------------------------------------------------
F1:        0.7084 ± 0.0382
Precision: 0.8155
Recall:   

## Experiments with AI

## HGB Probability Calibration + OOF Threshold Experiment

This experiment keeps the untouched test set separate. Calibration and threshold
selection use training OOF predictions only.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# Recreate the same leakage-free split and remove X37.
df_cal = pd.read_csv("../Data/polish_5year.csv")
df_cal = df_cal.drop(
    columns=["current_assets_minus_inventories_long_term_liabilities"]
)
X_cal = df_cal.drop(columns=["class"])
y_cal = df_cal["class"]

X_train_cal, X_test_cal, y_train_cal, y_test_cal = train_test_split(
    X_cal,
    y_cal,
    test_size=0.20,
    random_state=42,
    stratify=y_cal
)

cv_cal = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

base_hgb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

base_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", base_hgb)
])

calibration_results = []
calibrated_models = {}

for calibration_method in ["sigmoid", "isotonic"]:
    print("\n" + "=" * 70)
    print(f"HGB + {calibration_method.upper()} CALIBRATION")
    print("=" * 70)

    calibrated = CalibratedClassifierCV(
        estimator=base_pipeline,
        method=calibration_method,
        cv=cv_cal,
        n_jobs=-1,
        ensemble=True
    )
    calibrated_models[calibration_method] = calibrated

    oof_probability = cross_val_predict(
        calibrated,
        X_train_cal,
        y_train_cal,
        cv=cv_cal,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    threshold_rows = []
    for threshold in np.arange(0.05, 0.951, 0.01):
        oof_prediction = (oof_probability >= threshold).astype(int)
        threshold_rows.append({
            "Threshold": threshold,
            "F1": f1_score(y_train_cal, oof_prediction),
            "Precision": precision_score(
                y_train_cal, oof_prediction, zero_division=0
            ),
            "Recall": recall_score(
                y_train_cal, oof_prediction, zero_division=0
            )
        })

    threshold_df = pd.DataFrame(threshold_rows)
    best_row = threshold_df.loc[threshold_df["F1"].idxmax()]
    threshold = float(best_row["Threshold"])

    oof_roc_auc = roc_auc_score(y_train_cal, oof_probability)
    oof_pr_auc = average_precision_score(y_train_cal, oof_probability)

    print(f"OOF threshold: {threshold:.2f}")
    print(f"OOF F1:        {best_row['F1']:.4f}")
    print(f"OOF Precision: {best_row['Precision']:.4f}")
    print(f"OOF Recall:    {best_row['Recall']:.4f}")
    print(f"OOF ROC-AUC:   {oof_roc_auc:.4f}")
    print(f"OOF PR-AUC:    {oof_pr_auc:.4f}")

    calibration_results.append({
        "Calibration": calibration_method,
        "OOF Threshold": threshold,
        "OOF F1": float(best_row["F1"]),
        "OOF Precision": float(best_row["Precision"]),
        "OOF Recall": float(best_row["Recall"]),
        "OOF ROC-AUC": oof_roc_auc,
        "OOF PR-AUC": oof_pr_auc
    })

calibration_df = pd.DataFrame(calibration_results).sort_values(
    "OOF F1",
    ascending=False
)

print("\n" + "=" * 70)
print("HGB CALIBRATION - TRAINING OOF COMPARISON")
print("=" * 70)
print(
    calibration_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

# Select by OOF F1 only; the test set is not consulted for model/calibration choice.
selected_method = calibration_df.iloc[0]["Calibration"]
selected_threshold = float(calibration_df.iloc[0]["OOF Threshold"])
print(f"\nSelected method: {selected_method}")
print(f"Selected OOF threshold: {selected_threshold:.2f}")

selected_pipeline = calibrated_models[selected_method]
selected_pipeline.fit(X_train_cal, y_train_cal)
test_probability = selected_pipeline.predict_proba(X_test_cal)[:, 1]
test_prediction = (test_probability >= selected_threshold).astype(int)

test_f1 = f1_score(y_test_cal, test_prediction)
test_precision = precision_score(
    y_test_cal, test_prediction, zero_division=0
)
test_recall = recall_score(
    y_test_cal, test_prediction, zero_division=0
)
test_roc_auc = roc_auc_score(y_test_cal, test_probability)
test_pr_auc = average_precision_score(y_test_cal, test_probability)

print("\n" + "=" * 70)
print("HGB + CALIBRATION - UNTOUCHed TEST EVALUATION")
print("=" * 70)
print(f"Calibration: {selected_method}")
print(f"Threshold:   {selected_threshold:.2f}")
print(f"F1:          {test_f1:.4f}")
print(f"Precision:   {test_precision:.4f}")
print(f"Recall:      {test_recall:.4f}")
print(f"ROC-AUC:     {test_roc_auc:.4f}")
print(f"PR-AUC:      {test_pr_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_cal, test_prediction, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test_cal, test_prediction))

hgb_calibration_test_result = {
    "Model": f"HGB + {selected_method} calibration",
    "Test F1": test_f1,
    "Test Precision": test_precision,
    "Test Recall": test_recall,
    "Test ROC-AUC": test_roc_auc,
    "Test PR-AUC": test_pr_auc
}


## XGBoost + SMOTE OOF Threshold Experiment

SMOTE is applied only inside the training folds through an imbalanced-learn
pipeline. The OOF threshold is selected from training predictions only.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

df_smote = pd.read_csv("../Data/polish_5year.csv")
df_smote = df_smote.drop(
    columns=["current_assets_minus_inventories_long_term_liabilities"]
)
X_smote = df_smote.drop(columns=["class"])
y_smote = df_smote["class"]

X_train_smote, X_test_smote, y_train_smote, y_test_smote = train_test_split(
    X_smote,
    y_smote,
    test_size=0.20,
    random_state=42,
    stratify=y_smote
)

cv_smote = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

smote_xgb_pipeline = ImbPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("smote", SMOTE(random_state=42, k_neighbors=5)),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

oof_probability_smote = cross_val_predict(
    smote_xgb_pipeline,
    X_train_smote,
    y_train_smote,
    cv=cv_smote,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

threshold_rows = []
for threshold in np.arange(0.05, 0.951, 0.01):
    oof_prediction = (oof_probability_smote >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "F1": f1_score(y_train_smote, oof_prediction),
        "Precision": precision_score(
            y_train_smote, oof_prediction, zero_division=0
        ),
        "Recall": recall_score(
            y_train_smote, oof_prediction, zero_division=0
        )
    })

threshold_df_smote = pd.DataFrame(threshold_rows)
best_smote_row = threshold_df_smote.loc[
    threshold_df_smote["F1"].idxmax()
]
smote_threshold = float(best_smote_row["Threshold"])

oof_f1_smote = float(best_smote_row["F1"])
oof_precision_smote = float(best_smote_row["Precision"])
oof_recall_smote = float(best_smote_row["Recall"])
oof_roc_smote = roc_auc_score(y_train_smote, oof_probability_smote)
oof_pr_smote = average_precision_score(y_train_smote, oof_probability_smote)

smote_xgb_pipeline.fit(X_train_smote, y_train_smote)
test_probability_smote = smote_xgb_pipeline.predict_proba(
    X_test_smote
)[:, 1]
test_prediction_smote = (
    test_probability_smote >= smote_threshold
).astype(int)

test_f1_smote = f1_score(y_test_smote, test_prediction_smote)
test_precision_smote = precision_score(
    y_test_smote, test_prediction_smote, zero_division=0
)
test_recall_smote = recall_score(
    y_test_smote, test_prediction_smote, zero_division=0
)
test_roc_smote = roc_auc_score(y_test_smote, test_probability_smote)
test_pr_smote = average_precision_score(
    y_test_smote, test_probability_smote
)

print("\n" + "=" * 70)
print("XGBOOST + SMOTE - TRAINING OOF AND FINAL TEST")
print("=" * 70)
print(f"OOF threshold: {smote_threshold:.2f}")
print(f"OOF F1:        {oof_f1_smote:.4f}")
print(f"OOF Precision: {oof_precision_smote:.4f}")
print(f"OOF Recall:    {oof_recall_smote:.4f}")
print(f"OOF ROC-AUC:   {oof_roc_smote:.4f}")
print(f"OOF PR-AUC:    {oof_pr_smote:.4f}")
print("\nFinal test:")
print(f"F1:            {test_f1_smote:.4f}")
print(f"Precision:     {test_precision_smote:.4f}")
print(f"Recall:        {test_recall_smote:.4f}")
print(f"ROC-AUC:       {test_roc_smote:.4f}")
print(f"PR-AUC:        {test_pr_smote:.4f}")
print("\nClassification Report:")
print(
    classification_report(
        y_test_smote,
        test_prediction_smote,
        digits=4
    )
)
print("Confusion Matrix:")
print(confusion_matrix(y_test_smote, test_prediction_smote))

xgb_smote_test_result = {
    "Model": "XGBoost + SMOTE",
    "Test F1": test_f1_smote,
    "Test Precision": test_precision_smote,
    "Test Recall": test_recall_smote,
    "Test ROC-AUC": test_roc_smote,
    "Test PR-AUC": test_pr_smote
}

# Final leaderboard for the principal model comparisons.
leaderboard = pd.DataFrame([
    {"Model": "XGBoost", "CV F1": 0.6589, "Test F1": 0.6515},
    {"Model": "HistGradientBoosting", "CV F1": 0.6931, "Test F1": 0.6917},
    {"Model": "LightGBM baseline", "CV F1": 0.7063, "Test F1": 0.7172},
    {"Model": "Tuned LightGBM", "CV F1": 0.7130, "Test F1": 0.6861},
    {"Model": "LightGBM 15:1", "CV F1": 0.7109, "Test F1": 0.6901},
    {"Model": "HGB + sigmoid calibration", "CV F1": 0.7173, "Test F1": 0.6792},
    {"Model": "XGBoost + SMOTE", "CV F1": oof_f1_smote, "Test F1": test_f1_smote}
])

print("\n" + "=" * 70)
print("FINAL MODEL LEADERBOARD")
print("=" * 70)
print(
    leaderboard.sort_values("Test F1", ascending=False).to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

print("\nBEST MODEL:", leaderboard.sort_values(
    "Test F1", ascending=False
).iloc[0]["Model"])
print("BEST TEST F1:", f"{leaderboard['Test F1'].max():.4f}")


## New Experiment Results

- **HGB + sigmoid calibration:** OOF threshold `0.26`, OOF F1 `0.7173`, final test F1 `0.6792`, precision `0.7013`, recall `0.6585`, ROC-AUC `0.9608`, PR-AUC `0.7886`.
- **HGB + isotonic calibration:** OOF threshold `0.36`, OOF F1 `0.7129`, but it was not selected because it had lower OOF F1.
- **XGBoost + SMOTE:** OOF threshold `0.59`, OOF F1 `0.6055`, final test F1 `0.5732`, precision `0.6000`, recall `0.5488`, ROC-AUC `0.9192`, PR-AUC `0.6318`.

**Conclusion:** the original LightGBM baseline remains the best-performing model at final test F1 `0.7172`. Neither calibration nor SMOTE improved the untouched-test result.


## Controlled Financial Feature Engineering and OOF Stacking

This section evaluates a small set of meaningful derived financial features,
adds no more than the listed transformations, and uses 5-fold OOF predictions
for threshold and stacking decisions. The untouched test set is evaluated only
after OOF model selection.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)
from sklearn.preprocessing import FunctionTransformer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

FEATURE_BENCHMARK_F1 = 0.7172

df_fe = pd.read_csv("../Data/polish_5year.csv")
df_fe = df_fe.drop(
    columns=["current_assets_minus_inventories_long_term_liabilities"]
)
X_fe = df_fe.drop(columns=["class"])
y_fe = df_fe["class"]

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe,
    y_fe,
    test_size=0.20,
    random_state=42,
    stratify=y_fe
)

cv_fe = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


def add_controlled_financial_features(frame):
    column_names = X_fe.columns
    column_index = {name: index for index, name in enumerate(column_names)}
    numeric = np.asarray(frame, dtype=float)

    def safe_ratio(numerator, denominator):
        numerator_values = numeric[:, column_index[numerator]]
        denominator_values = numeric[:, column_index[denominator]]
        result = np.divide(
            numerator_values,
            denominator_values,
            out=np.zeros_like(numerator_values),
            where=np.abs(denominator_values) > 1e-12
        )
        return np.clip(result, -1e6, 1e6)
    
    derived = []

    # Controlled ratios: 6 transformations.
    derived.append(safe_ratio(
        "ebit_total_assets",
        "total_liabilities_total_assets"
    ))
    derived.append(safe_ratio(
        "ebitda_total_assets",
        "total_liabilities_total_assets"
    ))
    derived.append(
        numeric[:, column_index["gross_profit_sales"]]
        * numeric[:, column_index["sales_total_assets"]]
    )
    derived.append(
        numeric[:, column_index["net_profit_total_assets"]]
        * numeric[:, column_index["sales_total_assets"]]
    )
    derived.append(safe_ratio(
        "working_capital",
        "total_liabilities_total_assets"
    ))
    derived.append(
        numeric[:, column_index["equity_total_assets"]]
        * numeric[:, column_index["total_assets_total_liabilities"]]
    )

    # Controlled differences/interactions: 6 transformations.
    derived.append(
        numeric[:, column_index["net_profit_total_assets"]]
        - numeric[:, column_index["operating_profit_total_assets"]]
    )
    derived.append(
        numeric[:, column_index["current_assets_short_term_liabilities"]]
        - numeric[:, column_index["working_capital_total_assets"]]
    )
    derived.append(
        numeric[:, column_index["equity_total_assets"]]
        - numeric[:, column_index["total_liabilities_total_assets"]]
    )
    derived.append(
        numeric[:, column_index["ebitda_total_assets"]]
        * numeric[:, column_index["sales_total_assets"]]
    )
    derived.append(
        numeric[:, column_index["retained_earnings_total_assets"]]
        * numeric[:, column_index["total_assets_total_liabilities"]]
    )
    derived.append(
        numeric[:, column_index["operating_expenses_short_term_liabilities"]]
        - numeric[:, column_index["short_term_liabilities_total_assets"]]
    )

    numeric = np.column_stack([numeric] + derived)
    numeric = np.nan_to_num(
        numeric,
        nan=0.0,
        posinf=1e6,
        neginf=-1e6
    )
    return np.clip(numeric, -1e6, 1e6)


feature_engineering = FunctionTransformer(
    add_controlled_financial_features,
    validate=False
)

# 12 controlled derived features are appended to the 63 original inputs.
print("Controlled derived feature count: 12")
print("Original input count:", X_train_fe.shape[1])

xgb_engineered = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("features", feature_engineering),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

lgbm_engineered = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("features", feature_engineering),
    ("classifier", LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    ))
])

lgbm_selected = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("features", feature_engineering),
    ("selection", SelectKBest(score_func=f_classif, k=40)),
    ("classifier", LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42,
        verbosity=-1,
        n_jobs=-1
    ))
])

engineering_candidates = {
    "XGBoost + 12 features": xgb_engineered,
    "LightGBM + 12 features": lgbm_engineered,
    "LightGBM + SelectKBest 40": lgbm_selected
}

engineering_results = []
engineering_oof_probabilities = {}

print("\n" + "=" * 70)
print("CONTROLLED SYNTHETIC FEATURE ENGINEERING")
print("=" * 70)

for name, candidate in engineering_candidates.items():
    oof_probability = cross_val_predict(
        candidate,
        X_train_fe,
        y_train_fe,
        cv=cv_fe,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]
    engineering_oof_probabilities[name] = oof_probability

    threshold_rows = []
    for threshold in np.arange(0.05, 0.951, 0.01):
        prediction = (oof_probability >= threshold).astype(int)
        threshold_rows.append({
            "Threshold": threshold,
            "F1": f1_score(y_train_fe, prediction),
            "Precision": precision_score(
                y_train_fe, prediction, zero_division=0
            ),
            "Recall": recall_score(
                y_train_fe, prediction, zero_division=0
            )
        })

    threshold_table = pd.DataFrame(threshold_rows)
    best_row = threshold_table.loc[threshold_table["F1"].idxmax()]
    threshold = float(best_row["Threshold"])

    engineering_results.append({
        "Model": name,
        "OOF Threshold": threshold,
        "OOF F1": float(best_row["F1"]),
        "OOF Precision": float(best_row["Precision"]),
        "OOF Recall": float(best_row["Recall"]),
        "OOF ROC-AUC": roc_auc_score(y_train_fe, oof_probability),
        "OOF PR-AUC": average_precision_score(
            y_train_fe, oof_probability
        )
    })

engineering_results_df = pd.DataFrame(engineering_results).sort_values(
    "OOF F1",
    ascending=False
)
print(
    engineering_results_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

# OOF only is used to select the engineering candidate.
selected_engineering_name = engineering_results_df.iloc[0]["Model"]
selected_engineering_threshold = float(
    engineering_results_df.iloc[0]["OOF Threshold"]
)
selected_engineering_pipeline = engineering_candidates[
    selected_engineering_name
]
selected_engineering_pipeline.fit(X_train_fe, y_train_fe)

engineering_test_probability = selected_engineering_pipeline.predict_proba(
    X_test_fe
)[:, 1]
engineering_test_prediction = (
    engineering_test_probability >= selected_engineering_threshold
).astype(int)

engineering_test_f1 = f1_score(
    y_test_fe, engineering_test_prediction
)
engineering_test_precision = precision_score(
    y_test_fe, engineering_test_prediction, zero_division=0
)
engineering_test_recall = recall_score(
    y_test_fe, engineering_test_prediction, zero_division=0
)
engineering_test_roc_auc = roc_auc_score(
    y_test_fe, engineering_test_probability
)
engineering_test_pr_auc = average_precision_score(
    y_test_fe, engineering_test_probability
)

print("\nSelected engineered candidate:", selected_engineering_name)
print("Selected OOF threshold:", f"{selected_engineering_threshold:.2f}")
print("\nENGINEERED FEATURE FINAL TEST")
print(f"F1:        {engineering_test_f1:.4f}")
print(f"Precision: {engineering_test_precision:.4f}")
print(f"Recall:    {engineering_test_recall:.4f}")
print(f"ROC-AUC:   {engineering_test_roc_auc:.4f}")
print(f"PR-AUC:    {engineering_test_pr_auc:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test_fe, engineering_test_prediction))

engineered_improvement = (
    engineering_test_f1 > FEATURE_BENCHMARK_F1
)

stacking_test_f1 = None

if not engineered_improvement:
    print("\n" + "=" * 70)
    print("STACKING ENSEMBLE (ENGINEERING DID NOT BEAT BENCHMARK)")
    print("=" * 70)

    base_pipelines = {
        "HGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", HistGradientBoostingClassifier(
                max_iter=300,
                learning_rate=0.05,
                max_leaf_nodes=31,
                random_state=42
            ))
        ]),
        "LightGBM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", LGBMClassifier(
                n_estimators=300,
                learning_rate=0.05,
                num_leaves=31,
                max_depth=-1,
                subsample=0.8,
                colsample_bytree=0.8,
                class_weight="balanced",
                random_state=42,
                verbosity=-1,
                n_jobs=-1
            ))
        ]),
        "XGBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("classifier", XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            ))
        ])
    }

    oof_columns = []
    test_columns = []

    for base_name, base_pipeline in base_pipelines.items():
        oof_prob = cross_val_predict(
            base_pipeline,
            X_train_fe,
            y_train_fe,
            cv=cv_fe,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]
        oof_columns.append(oof_prob)

        base_pipeline.fit(X_train_fe, y_train_fe)
        test_columns.append(
            base_pipeline.predict_proba(X_test_fe)[:, 1]
        )

    oof_meta_features = np.column_stack(oof_columns)
    test_meta_features = np.column_stack(test_columns)

    meta_model = Pipeline([
        ("scaler", StandardScaler()),
        ("meta", LogisticRegression(
            random_state=42,
            max_iter=2000
        ))
    ])
    meta_model.fit(oof_meta_features, y_train_fe)

    stacking_oof_probability = meta_model.predict_proba(
        oof_meta_features
    )[:, 1]

    stacking_threshold_rows = []
    for threshold in np.arange(0.05, 0.951, 0.01):
        prediction = (
            stacking_oof_probability >= threshold
        ).astype(int)
        stacking_threshold_rows.append({
            "Threshold": threshold,
            "F1": f1_score(y_train_fe, prediction),
            "Precision": precision_score(
                y_train_fe, prediction, zero_division=0
            ),
            "Recall": recall_score(
                y_train_fe, prediction, zero_division=0
            )
        })

    stacking_threshold_table = pd.DataFrame(
        stacking_threshold_rows
    )
    stacking_best_row = stacking_threshold_table.loc[
        stacking_threshold_table["F1"].idxmax()
    ]
    stacking_threshold = float(stacking_best_row["Threshold"])

    stacking_oof_f1 = float(stacking_best_row["F1"])
    stacking_oof_precision = float(stacking_best_row["Precision"])
    stacking_oof_recall = float(stacking_best_row["Recall"])
    stacking_oof_roc_auc = roc_auc_score(
        y_train_fe, stacking_oof_probability
    )
    stacking_oof_pr_auc = average_precision_score(
        y_train_fe, stacking_oof_probability
    )

    stacking_test_probability = meta_model.predict_proba(
        test_meta_features
    )[:, 1]
    stacking_test_prediction = (
        stacking_test_probability >= stacking_threshold
    ).astype(int)

    stacking_test_f1 = f1_score(
        y_test_fe, stacking_test_prediction
    )
    stacking_test_precision = precision_score(
        y_test_fe, stacking_test_prediction, zero_division=0
    )
    stacking_test_recall = recall_score(
        y_test_fe, stacking_test_prediction, zero_division=0
    )
    stacking_test_roc_auc = roc_auc_score(
        y_test_fe, stacking_test_probability
    )
    stacking_test_pr_auc = average_precision_score(
        y_test_fe, stacking_test_probability
    )

    print(f"OOF threshold: {stacking_threshold:.2f}")
    print(f"OOF F1:        {stacking_oof_f1:.4f}")
    print(f"OOF Precision: {stacking_oof_precision:.4f}")
    print(f"OOF Recall:    {stacking_oof_recall:.4f}")
    print(f"OOF ROC-AUC:   {stacking_oof_roc_auc:.4f}")
    print(f"OOF PR-AUC:    {stacking_oof_pr_auc:.4f}")
    print("\nSTACKING FINAL TEST")
    print(f"F1:        {stacking_test_f1:.4f}")
    print(f"Precision: {stacking_test_precision:.4f}")
    print(f"Recall:    {stacking_test_recall:.4f}")
    print(f"ROC-AUC:   {stacking_test_roc_auc:.4f}")
    print(f"PR-AUC:    {stacking_test_pr_auc:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test_fe, stacking_test_prediction))

# Update the leaderboard with only actual completed results.
leaderboard_final = pd.DataFrame([
    {"Model": "XGBoost", "CV/OOF F1": 0.6589, "Test F1": 0.6515},
    {"Model": "HistGradientBoosting", "CV/OOF F1": 0.6931, "Test F1": 0.6917},
    {"Model": "LightGBM baseline", "CV/OOF F1": 0.7063, "Test F1": 0.7172},
    {"Model": "Tuned LightGBM", "CV/OOF F1": 0.7130, "Test F1": 0.6861},
    {"Model": "LightGBM 15:1", "CV/OOF F1": 0.7109, "Test F1": 0.6901},
    {"Model": "HGB + sigmoid calibration", "CV/OOF F1": 0.7173, "Test F1": 0.6792},
    {"Model": "XGBoost + SMOTE", "CV/OOF F1": 0.6055, "Test F1": 0.5732},
    {
        "Model": selected_engineering_name,
        "CV/OOF F1": float(engineering_results_df.iloc[0]["OOF F1"]),
        "Test F1": engineering_test_f1
    }
])

if stacking_test_f1 is not None:
    leaderboard_final = pd.concat(
        [
            leaderboard_final,
            pd.DataFrame([{
                "Model": "HGB + LightGBM + XGBoost Logistic Stack",
                "CV/OOF F1": stacking_oof_f1,
                "Test F1": stacking_test_f1
            }])
        ],
        ignore_index=True
    )

print("\n" + "=" * 70)
print("UPDATED FINAL LEADERBOARD")
print("=" * 70)
print(
    leaderboard_final.sort_values(
        "Test F1", ascending=False
    ).to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

winner = leaderboard_final.sort_values(
    "Test F1", ascending=False
).iloc[0]
print("\nBEST MODEL:", winner["Model"])
print("BEST TEST F1:", f"{winner['Test F1']:.4f}")


## Final Feature Engineering and Stacking Results

### Controlled synthetic feature engineering
The experiment added 12 clipped, finite derived features:
- 6 ratios
- 6 differences/products/interactions

| Candidate | OOF Threshold | OOF F1 | Test F1 | Test Precision | Test Recall | Test ROC-AUC | Test PR-AUC |
|---|---:|---:|---:|---:|---:|---:|---:|
| LightGBM + 12 features | 0.63 | 0.7063 | 0.6861 | 0.8545 | 0.5732 | 0.9564 | 0.7965 |
| XGBoost + 12 features | 0.31 | 0.7033 | - | - | - | - | - |
| LightGBM + SelectKBest 40 | 0.24 | 0.5135 | - | - | - | - | - |

The LightGBM engineered candidate reached the tuned-LightGBM test result but did
not improve the original LightGBM baseline.

### Stacking ensemble
Because engineered features did not beat the benchmark, an OOF logistic
stacking ensemble was trained from HGB, LightGBM, and XGBoost probability
predictions.

- OOF threshold: `0.22`
- OOF F1: `0.7218`
- Test F1: `0.7027`
- Test precision: `0.7879`
- Test recall: `0.6341`
- Test ROC-AUC: `0.9516`
- Test PR-AUC: `0.7740`
- Confusion matrix: `[[1086, 14], [30, 52]]`

### Conclusion
**The original LightGBM baseline remains the best model.**

- Best Test F1: `0.7172`
- Synthetic feature engineering: no improvement
- OOF stacking: no improvement

The next research direction should focus on stronger data-specific approaches
such as calibrated nonlinear ensembles or deeper feature-selection experiments,
not random model additions.
